In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("IMDB.csv")
df.head()
print(len(df))

50000


In [4]:
df['label'] = 1
df.loc[df['sentiment'] == 'negative', 'label'] = 0
df['review_clean'] = df['review'].str.replace(r'<.*?>', ' ', regex=True)

In [5]:
df.head()

,review,sentiment,label,review_clean
0,One of the other reviewers has mentioned that ...,positive,1,One of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,1,A wonderful little production. The filming t...
2,I thought this was a wonderful way to spend ti...,positive,1,I thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,0,Basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1,"Petter Mattei's ""Love in the Time of Money"" is..."


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=20000,
    min_df=5,
    max_df=0.9,
    ngram_range=(1, 2)
)
X = list(vectorizer.fit_transform(df['review_clean'].tolist()))
#X = (vectorizer.fit_transform(df['review_clean']))

vectorizer.get_feature_names_out()

array(['00', '000', '000 000', ..., 'zu', 'zucco', 'zucker'],
      shape=(20000,), dtype=object)

In [7]:
len(df['review_clean']), len(df['label'])

(50000, 50000)

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, list(df['label']), test_size=0.2, random_state=42)

In [9]:
type(X_train)

list

In [10]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [11]:
device

'cpu'

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
X = vectorizer.fit_transform(df['review_clean'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))

0.898


In [13]:
from sklearn.neural_network import MLPClassifier
model = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(100, 16), random_state=42)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))

0.8841


/home/nel/PycharmProjects/learning_nlp/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [14]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(20000, 256),
            nn.Softmax(dim=1),
            nn.Linear(256, 1),
            nn.Softmax(dim=1)
        ).to(device)

    def forward(self, x):
        return self.model(x)

from torch.utils.data import Dataset, DataLoader
class TfidDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].toarray().squeeze(0)
        x = torch.tensor(x, dtype=torch.float).to(device)

        label = torch.tensor(self.y.iloc[idx], dtype=torch.float).to(device)
        return x, label


input_dim = X_train.shape[1]
model = MLP().to(device)







In [15]:
from tqdm import tqdm
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 5

train_dataset = TfidDataset(X_train, y_train)
test_dataset = TfidDataset(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)



for epoch in tqdm(range(epochs)):
    model.train()
    total_loss = 0

    for X, y in train_dataloader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X).squeeze(1)
        loss = loss_fn(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1}, Loss: {avg_loss}")

 20%|██        | 1/5 [00:44<02:57, 44.28s/it]

Epoch 1, Loss: 0.0


 40%|████      | 2/5 [01:32<02:19, 46.66s/it]

Epoch 2, Loss: 0.0


 60%|██████    | 3/5 [02:16<01:30, 45.34s/it]

Epoch 3, Loss: 0.0


 80%|████████  | 4/5 [02:59<00:44, 44.37s/it]

Epoch 4, Loss: 0.0


100%|██████████| 5/5 [03:46<00:00, 45.26s/it]

Epoch 5, Loss: 0.0


In [16]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for X, y in test_dataloader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X).squeeze(1)   # shape: [batch_size]
        probs = torch.sigmoid(logits)  # convert logits to probabilities

        preds = (probs >= 0.5).float() # 1 if positive, 0 if negative

        correct += (preds == y).sum().item()
        total += y.size(0)

accuracy = correct / total
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.5039
